In [0]:
%pip install --quiet pandas openpyxl
dbutils.library.restartPython()  


In [0]:
%pip install xlrd

import pandas as pd

df = pd.read_excel(
    "/Volumes/my_catalog1/orders_db/order_excel/Sample - Superstore.xls",
    header=0
)

df.columns = [
    col.strip().replace(" ", "_")
    .replace(",", "_")
    .replace(";", "_")
    .replace("{", "_")
    .replace("}", "_")
    .replace("(", "_")
    .replace(")", "_")
    .replace("\n", "_")
    .replace("\t", "_")
    .replace("=", "_")
    for col in df.columns
]

spark_df = spark.createDataFrame(df)

spark_df.write.mode("overwrite").saveAsTable("my_catalog1.orders_db.superstore_table")

CREATING DIMENSION AND FACT TABLES

In [0]:
customer_columns = [
    "Customer_ID", "Customer_Name", "Segment", "Country", "City", "State", "Postal_Code", "Region"
]
customer_df = (
    spark_df.select(*customer_columns)
    .filter("Customer_ID IS NOT NULL")
    .dropDuplicates(["Customer_ID"])
)
customer_df.write.mode("overwrite").saveAsTable("my_catalog1.orders_db.customer_dim")


product_columns = [
    "Product_ID", "Product_Name", "Category", "Sub-Category"
]
product_df = (
    spark_df.select(*product_columns)
    .filter("Product_ID IS NOT NULL")
    .dropDuplicates(["Product_ID"])
)
product_df.write.mode("overwrite").saveAsTable("my_catalog1.orders_db.product_dim")


sales_fact_columns = [
    "Order_ID", "Order_Date", "Ship_Date", "Ship_Mode", "Sales", "Quantity", "Discount", "Profit",
    "Customer_ID", "Product_ID"
]
sales_fact_df = (
    spark_df.select(*sales_fact_columns)
    .filter("Order_ID IS NOT NULL")
)
sales_fact_df.write.mode("overwrite").saveAsTable("my_catalog1.orders_db.sales_fact")

spark.sql("""
ALTER TABLE my_catalog1.orders_db.customer_dim
DROP PRIMARY KEY CASCADE
""")

spark.sql("""
ALTER TABLE my_catalog1.orders_db.product_dim
DROP PRIMARY KEY CASCADE
""")

spark.sql("""
ALTER TABLE my_catalog1.orders_db.sales_fact
DROP PRIMARY KEY CASCADE
""")


spark.sql("""
ALTER TABLE my_catalog1.orders_db.customer_dim
ADD CONSTRAINT customer_pk PRIMARY KEY (Customer_ID)
""")

spark.sql("""
ALTER TABLE my_catalog1.orders_db.product_dim
ADD CONSTRAINT product_pk PRIMARY KEY (Product_ID)
""")

spark.sql("""
ALTER TABLE my_catalog1.orders_db.sales_fact
ADD CONSTRAINT sales_fact_pk PRIMARY KEY (Order_ID)
""")

spark.sql("""
ALTER TABLE my_catalog1.orders_db.sales_fact
ADD CONSTRAINT fk_customer FOREIGN KEY (Customer_ID) REFERENCES my_catalog1.orders_db.customer_dim(Customer_ID)
""")

spark.sql("""
ALTER TABLE my_catalog1.orders_db.sales_fact
ADD CONSTRAINT fk_product FOREIGN KEY (Product_ID) REFERENCES my_catalog1.orders_db.product_dim(Product_ID)
""")